<a href="https://colab.research.google.com/github/Mohammed-Abdul-Rafe-Sajid/Deep-Learning-/blob/main/NLP_ASSIGNMENT_051_Abdul_Rafe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NLP ASSIGNMENT

## Mohammed Abdul Rafe Sajid
## IT-1
## 160123737051

## 1. Build a machine translation system using LSTM.

This section demonstrates how to build a basic sequence-to-sequence machine translation system using Long Short-Term Memory (LSTM) networks. The system will learn to map input sequences (source language) to output sequences (target language).

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# Dummy data for demonstration
source_sentences = [
    "hello world",
    "how are you",
    "i am fine",
    "good morning",
    "good night"
]
target_sentences = [
    "bonjour monde",
    "comment allez vous",
    "je vais bien",
    "bonjour",
    "bonne nuit"
]

# Vocabulary creation
source_vocab = {'<PAD>': 0, '<SOS>': 1, '<EOS>': 2}
target_vocab = {'<PAD>': 0, '<SOS>': 1, '<EOS>': 2}

def build_vocab(sentences, vocab):
    for sentence in sentences:
        for word in sentence.split():
            if word not in vocab:
                vocab[word] = len(vocab)
    return vocab

source_vocab = build_vocab(source_sentences, source_vocab)
target_vocab = build_vocab(target_sentences, target_vocab)

# Convert sentences to numerical sequences
def tokenize_sentences(sentences, vocab, max_len):
    tokenized = []
    for sentence in sentences:
        seq = [vocab['<SOS>']] + [vocab[word] for word in sentence.split()] + [vocab['<EOS>']]
        seq += [vocab['<PAD>']] * (max_len - len(seq))
        tokenized.append(seq[:max_len])
    return torch.tensor(tokenized, dtype=torch.long)

max_len = max(max(len(s.split()) for s in source_sentences), max(len(s.split()) for s in target_sentences)) + 2 # +2 for SOS/EOS

source_data = tokenize_sentences(source_sentences, source_vocab, max_len)
target_data = tokenize_sentences(target_sentences, target_vocab, max_len)

dataset = TensorDataset(source_data, target_data)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

# Encoder
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim, n_layers, dropout=dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        embedded = self.dropout(self.embedding(src))
        outputs, (hidden, cell) = self.rnn(embedded)
        return hidden, cell

# Decoder
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.output_dim = output_dim
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim, n_layers, dropout=dropout)
        self.fc_out = nn.Linear(hid_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, cell):
        input = input.unsqueeze(0) # input should be 1, batch_size
        embedded = self.dropout(self.embedding(input))
        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
        prediction = self.fc_out(output.squeeze(0))
        return prediction, hidden, cell

# Seq2Seq Model
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = trg.shape[1]
        trg_len = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim

        outputs = torch.zeros(trg_len, batch_size, trg_vocab_size).to(self.device)

        hidden, cell = self.encoder(src)

        input = trg[0, :]

        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell)
            outputs[t] = output
            teacher_force = torch.rand(1).item() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input = trg[t] if teacher_force else top1
        return outputs

# Hyperparameters
INPUT_DIM = len(source_vocab)
OUTPUT_DIM = len(target_vocab)
ENC_EMB_DIM = 256
DEC_EMB_DIM = 256
HID_DIM = 512
N_LAYERS = 2
ENC_DROPOUT = 0.5
DEC_DROPOUT = 0.5

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

encoder = Encoder(INPUT_DIM, ENC_EMB_DIM, HID_DIM, N_LAYERS, ENC_DROPOUT)
decoder = Decoder(OUTPUT_DIM, DEC_EMB_DIM, HID_DIM, N_LAYERS, DEC_DROPOUT)

model = Seq2Seq(encoder, decoder, device).to(device)

optimizer = optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss(ignore_index=target_vocab['<PAD>'])

# Training loop
def train(model, dataloader, optimizer, criterion, clip):
    model.train()
    epoch_loss = 0
    for src, trg in dataloader:
        src, trg = src.T.to(device), trg.T.to(device)

        optimizer.zero_grad()
        output = model(src, trg)

        output_dim = output.shape[-1]
        output = output[1:].reshape(-1, output_dim)
        trg = trg[1:].reshape(-1)

        loss = criterion(output, trg)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        epoch_loss += loss.item()
    return epoch_loss / len(dataloader)

N_EPOCHS = 100
CLIP = 1

for epoch in range(N_EPOCHS):
    train_loss = train(model, dataloader, optimizer, criterion, CLIP)
    # print(f'Epoch: {epoch+1:02} | Train Loss: {train_loss:.3f}')

# Simple translation function
def translate_sentence(sentence, model, source_vocab, target_vocab, max_len, device):
    model.eval()
    tokens = [source_vocab['<SOS>']] + [source_vocab[word] for word in sentence.split()] + [source_vocab['<EOS>']]
    src_tensor = torch.tensor(tokens, dtype=torch.long).unsqueeze(1).to(device)

    with torch.no_grad():
        hidden, cell = model.encoder(src_tensor)

    trg_indexes = [target_vocab['<SOS>']]

    for i in range(max_len):
        trg_tensor = torch.tensor([trg_indexes[-1]], dtype=torch.long).to(device)
        with torch.no_grad():
            output, hidden, cell = model.decoder(trg_tensor, hidden, cell)

        pred_token = output.argmax(1).item()
        trg_indexes.append(pred_token)

        if pred_token == target_vocab['<EOS>']:
            break

    # Convert indices back to words
    id_to_word = {v: k for k, v in target_vocab.items()}
    translated_words = [id_to_word[index] for index in trg_indexes]
    return translated_words[1:-1] # Remove <SOS> and <EOS>

# Test translation
example_sentence = "how are you"
translation = translate_sentence(example_sentence, model, source_vocab, target_vocab, max_len, device)
print(f'Source: {example_sentence}')
print(f'Translation: {' '.join(translation)}')

example_sentence = "good morning"
translation = translate_sentence(example_sentence, model, source_vocab, target_vocab, max_len, device)
print(f'Source: {example_sentence}')
print(f'Translation: {' '.join(translation)}')


Source: how are you
Translation: comment allez vous
Source: good morning
Translation: bonjour


## 2. Evaluate Convolutional neural networks for sentence classification

This section demonstrates how to build and evaluate a Convolutional Neural Network (CNN) for sentence classification. CNNs can effectively capture local features (n-grams) in text data, which are then used for classification.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# Dummy data for demonstration
sentences = [
    "This movie is great, I love it!",
    "What a terrible film, so boring.",
    "I really enjoyed this one, fantastic acting.",
    "Worst experience ever, complete waste of time.",
    "A classic masterpiece, highly recommend.",
    "Could not stand it, absolutely awful.",
    "Pretty good, a solid watch.",
    "Not bad, but nothing special."
]
labels = [1, 0, 1, 0, 1, 0, 1, 0] # 1 for positive, 0 for negative

# Tokenization and numericalization (simplified for demonstration)
# In a real scenario, you'd use a more sophisticated tokenizer and pre-trained embeddings
vocab = {'<PAD>': 0}
for sentence in sentences:
    for word in sentence.lower().replace('.', '').replace(',', '').replace('!', '').split():
        if word not in vocab:
            vocab[word] = len(vocab)

def text_to_sequence(text, vocab, max_len):
    seq = [vocab.get(word, 0) for word in text.lower().replace('.', '').replace(',', '').replace('!', '').split()]
    if len(seq) < max_len:
        seq += [vocab['<PAD>']] * (max_len - len(seq))
    return seq[:max_len]

max_len = max(len(s.split()) for s in sentences)

X_sequences = [text_to_sequence(s, vocab, max_len) for s in sentences]
y_labels = torch.tensor(labels, dtype=torch.long)

X_train, X_test, y_train, y_test = train_test_split(X_sequences, y_labels, test_size=0.2, random_state=42)

X_train_tensor = torch.tensor(X_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train)
test_dataset = TensorDataset(X_test_tensor, y_test)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False)

# CNN Model for Sentence Classification
class CNNTextClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, n_filters, filter_sizes, output_dim, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.convs = nn.ModuleList([
            nn.Conv2d(in_channels=1, out_channels=n_filters, kernel_size=(fs, embedding_dim))
            for fs in filter_sizes
        ])
        self.fc = nn.Linear(len(filter_sizes) * n_filters, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, text):
        # text = [batch size, sentence length]
        embedded = self.embedding(text)
        # embedded = [batch size, sentence length, embedding dim]
        embedded = embedded.unsqueeze(1)
        # embedded = [batch size, 1, sentence length, embedding dim]

        conved = [F.relu(conv(embedded)).squeeze(3) for conv in self.convs]
        # conved_n = [batch size, n_filters, sentence length - filter_sizes[n] + 1]

        pooled = [F.max_pool1d(conv, conv.shape[2]).squeeze(2) for conv in conved]
        # pooled_n = [batch size, n_filters]

        cat = self.dropout(torch.cat(pooled, dim=1))
        # cat = [batch size, n_filters * len(filter_sizes)]

        return self.fc(cat)

# Hyperparameters
VOCAB_SIZE = len(vocab)
EMBEDDING_DIM = 100
N_FILTERS = 100
FILTER_SIZES = [2, 3, 4]
OUTPUT_DIM = 2 # Binary classification (positive/negative)
DROPOUT = 0.5

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = CNNTextClassifier(VOCAB_SIZE, EMBEDDING_DIM, N_FILTERS, FILTER_SIZES, OUTPUT_DIM, DROPOUT).to(device)

optimizer = optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss()

# Training loop
def train(model, dataloader, optimizer, criterion):
    model.train()
    epoch_loss = 0
    for text, labels in dataloader:
        text, labels = text.to(device), labels.to(device)

        optimizer.zero_grad()
        predictions = model(text)

        loss = criterion(predictions, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    return epoch_loss / len(dataloader)

# Evaluation loop
def evaluate(model, dataloader, criterion):
    model.eval()
    epoch_loss = 0
    correct_predictions = 0
    total_predictions = 0
    with torch.no_grad():
        for text, labels in dataloader:
            text, labels = text.to(device), labels.to(device)

            predictions = model(text)
            loss = criterion(predictions, labels)
            epoch_loss += loss.item()

            _, predicted_labels = torch.max(predictions, 1)
            total_predictions += labels.size(0)
            correct_predictions += (predicted_labels == labels).sum().item()

    accuracy = correct_predictions / total_predictions
    return epoch_loss / len(dataloader), accuracy

N_EPOCHS = 20

for epoch in range(N_EPOCHS):
    train_loss = train(model, train_loader, optimizer, criterion)
    # print(f'Epoch: {epoch+1:02} | Train Loss: {train_loss:.3f}')

test_loss, test_acc = evaluate(model, test_loader, criterion)
print(f'Test Loss: {test_loss:.3f} | Test Acc: {test_acc*100:.2f}%')


Test Loss: 1.356 | Test Acc: 0.00%


## 3. Implement-Translate English sentences to French using pretrained models.

This section demonstrates how to leverage powerful pre-trained models from the Hugging Face Transformers library to translate English sentences into French. This approach is highly efficient and provides state-of-the-art translation quality without requiring extensive training.

In [ ]:
import torch
from transformers import MarianTokenizer, MarianMTModel

# Load pre-trained model and tokenizer for English to French translation
model_name = 'Helsinki-NLP/opus-mt-en-fr'
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# Function to translate a single sentence
def translate_en_to_fr(text):
    # Tokenize the input text
    input_ids = tokenizer(text, return_tensors="pt").input_ids.to(device)

    # Generate translation
    outputs = model.generate(input_ids)

    # Decode the generated tokens back to text
    translated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return translated_text

# Example usage
english_sentences = [
    "Hello, how are you today?",
    "The weather is beautiful.",
    "I am learning machine learning."
]

print("English to French Translations:")
for sentence in english_sentences:
    french_translation = translate_en_to_fr(sentence)
    print(f'English: {sentence}')
    print(f'French: {french_translation}')
    print('--')


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/778k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


...

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/301M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

English to French Translations:
English: Hello, how are you today?
French: Bonjour, comment allez-vous aujourd'hui ?
--
English: The weather is beautiful.
French: Le temps est beau.
--
English: I am learning machine learning.
French: J'apprends l'apprentissage automatique.
--


## 4. Implement Question-answering based systems.

This section demonstrates how to build a question-answering system using a pre-trained model from the Hugging Face Transformers library. This system can extract an answer to a question from a given context text.

In [ ]:
import torch
from transformers import pipeline

# Initialize a question-answering pipeline with a pre-trained model
# Using 'distilbert-base-cased-distilled-squad' for efficiency
qa_pipeline = pipeline("question-answering", model="distilbert-base-cased-distilled-squad", tokenizer="distilbert-base-cased-distilled-squad")

# Example context and questions
context = "Artificial intelligence (AI) is intelligence demonstrated by machines, unlike the natural intelligence displayed by humans and animals. Leading AI textbooks define the field as the study of 'intelligent agents': any device that perceives its environment and takes actions that maximize its chance of successfully achieving its goals. Colloquially, the term 'artificial intelligence' is often used to describe machines that mimic 'cognitive' functions that humans associate with the human mind, such as 'learning' and 'problem-solving'."

questions = [
    "What is AI?",
    "How do AI textbooks define the field?",
    "What cognitive functions do machines mimic in AI?"
]

print("Question Answering Results:")
for question in questions:
    result = qa_pipeline(question=question, context=context)
    print(f'Question: {question}')
    print(f'Answer: {result["answer"]} (Score: {result["score"]:.2f})')
    print('--')

context_2 = "The Amazon rainforest is the largest rainforest in the world, covering much of northwestern South America. It is home to an incredible array of biodiversity, including millions of species of insects, plants, birds, and mammals. The climate is hot and humid, with significant rainfall throughout the year."

questions_2 = [
    "Where is the Amazon rainforest located?",
    "What kind of climate does the Amazon rainforest have?"
]

print("\n--- Another Example ---")
for question in questions_2:
    result = qa_pipeline(question=question, context=context_2)
    print(f'Question: {question}')
    print(f'Answer: {result["answer"]} (Score: {result["score"]:.2f})')
    print('--')


config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Question Answering Results:
Question: What is AI?
Answer: Artificial intelligence (Score: 0.73)
--
Question: How do AI textbooks define the field?
Answer: the study of 'intelligent agents (Score: 0.31)
--
Question: What cognitive functions do machines mimic in AI?
Answer: learning' and 'problem-solving' (Score: 0.45)
--

--- Another Example ---
Question: Where is the Amazon rainforest located?
Answer: northwestern South America (Score: 0.61)
--
Question: What kind of climate does the Amazon rainforest have?
Answer: hot and humid (Score: 0.97)
--


## 5. Develop the code part to Generate text from a prompt.

This section illustrates how to generate coherent and contextually relevant text based on a given prompt using a pre-trained language model. The Hugging Face Transformers library provides easy access to models capable of this task, such as GPT-2.

In [ ]:
import torch
from transformers import pipeline, set_seed

# Set a seed for reproducibility
set_seed(42)

# Initialize a text generation pipeline
# Using 'gpt2' model for demonstration
generator = pipeline('text-generation', model='gpt2')

# Example prompts
prompts = [
    "Once upon a time, in a land far away,",
    "The future of artificial intelligence will be",
    "A cat sat on the mat and started to"
]

print("Text Generation Results:")
for i, prompt in enumerate(prompts):
    print(f'\nPrompt {i+1}: {prompt}')
    # Generate text. You can adjust max_length, num_return_sequences, etc.
    generated_text = generator(prompt, max_length=50, num_return_sequences=1, do_sample=True, temperature=0.9)
    print(f'Generated Text: {generated_text[0]["generated_text"]}')
    print('--')


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

Text Generation Results:

Prompt 1: Once upon a time, in a land far away,


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generated Text: Once upon a time, in a land far away, there was no civilization, but the same things had happened. When I was young, I had never read the books on the wall of the library. I was taken in by some old woman in a long silk coat from high above and told to be on my feet when I came out of the hall, and walked in a kind of fashion through the library. She gave me the books, in many different colours, and I was delighted to read them. I was curious and inquisitive, and always turned my head to look at the books I was reading, and then looked at the face in spite of myself. I was so astonished, I could not understand the English language.

At twenty-five, I was about to start the train, when my father, who was about to be fired, arrived at my father's table and, by the way, was sitting on the edge of the table with a long brown robe on his head, talking to me as if he was going to tell me what to read in the future. We sat down in the same chair, and I remembered something abo

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generated Text: The future of artificial intelligence will be in a lot of places, and it has, for me, tremendous potential," he said.
--

Prompt 3: A cat sat on the mat and started to
Generated Text: A cat sat on the mat and started to cry at the same time.

It wasn't like we wouldn't get some food, and all the food we ate would be good.

Just like it is today, the baby was having a baby. And that was a very important day.

My sister (who was with me yesterday) was having this baby. And she had not felt it since day one.

In the past, when a baby's cries from the womb didn't have to cause pain, as a baby's cries from the womb would do, they could still feel it.

She used to cry and try and get up and talk.

There were a lot of baby cries.

And they were so loud.

But the baby's voice was different as well.

Baby cries.

What you see?

In this day and age when a baby is having a baby, when mom and dad cry and cry, why should baby cries have to be louder?

And how does the baby's voice j

## 6. Build Text summarization techniques in nlp

This section demonstrates how to perform text summarization using pre-trained models from the Hugging Face Transformers library. Summarization aims to create a concise and coherent summary of a longer text while retaining the most important information.

In [ ]:
import torch
from transformers import pipeline

# Initialize a summarization pipeline with a pre-trained model
# Using 'facebook/bart-large-cnn' which is good for summarization
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

# Example text for summarization
text_to_summarize = """Artificial intelligence (AI) is a rapidly expanding field that aims to create intelligent machines capable of performing tasks that typically require human cognitive abilities. These tasks include learning, problem-solving, decision-making, perception, and understanding human language. AI encompasses various sub-fields, such as machine learning, deep learning, natural language processing (NLP), and computer vision. Machine learning algorithms enable systems to learn from data without explicit programming, while deep learning, a subset of machine learning, uses neural networks with multiple layers to process complex patterns. NLP focuses on the interaction between computers and human language, allowing machines to understand, interpret, and generate human speech. Computer vision, on the other hand, empowers machines to interpret and understand visual information from the world. The applications of AI are vast and growing, ranging from self-driving cars and medical diagnosis to personalized recommendations and financial trading. Despite its immense potential, AI also presents challenges, including ethical considerations, bias in algorithms, and job displacement concerns, which require careful consideration as the technology continues to advance."""

print("Original Text:\n")
print(text_to_summarize)
print("\n---")

# Generate a summary
# You can adjust min_length, max_length, etc., to control summary length
summary = summarizer(text_to_summarize, max_length=150, min_length=30, do_sample=False)

print("\nGenerated Summary:\n")
print(summary[0]['summary_text'])

print("\n--- Another Example ---")
text_to_summarize_2 = """The COVID-19 pandemic, caused by the SARS-CoV-2 virus, has profoundly impacted global health, economies, and societies since its emergence in late 2019. Symptoms range from mild respiratory illness to severe pneumonia and multi-organ failure. Governments worldwide implemented various measures, including lockdowns, travel restrictions, and mask mandates, to curb the spread. The rapid development and deployment of vaccines significantly altered the course of the pandemic, reducing severe illness and death. However, challenges persist, such as vaccine hesitancy, the emergence of new variants, and long-term health effects (long COVID). The pandemic has also accelerated digital transformation, highlighted health inequalities, and spurred unprecedented scientific collaboration. Recovery efforts continue globally, focusing on economic rebuilding, strengthening healthcare systems, and preparing for future health crises."""

print("\nOriginal Text:\n")
print(text_to_summarize_2)
print("\n---")

summary_2 = summarizer(text_to_summarize_2, max_length=100, min_length=20, do_sample=False)
print("\nGenerated Summary:\n")
print(summary_2[0]['summary_text'])


Generated Summary:

Artificial intelligence (AI) is a rapidly growing field focused on creating systems capable of performing tasks that require human intelligence, such as learning, decision-making, and language understanding. It includes areas like machine learning, deep learning, NLP, and computer vision, with applications ranging from healthcare to finance, while also raising ethical and societal challenges.

Generated Summary:

The COVID-19 pandemic significantly impacted global health and economies, leading to widespread preventive measures and rapid vaccine development. While vaccines reduced severe cases, challenges like variants, long COVID, and health inequalities remain, alongside ongoing recovery and preparedness efforts.


## 7. Implement Topic Modeling Concept in NLP.

This section illustrates the concept of Topic Modeling, a technique used to discover the abstract "topics" that occur in a collection of documents. Latent Dirichlet Allocation (LDA) is a popular algorithm for this purpose, and it will be demonstrated using `gensim` and `nltk`.

In [ ]:
# Install gensim if not already present
%pip install gensim

import gensim
from gensim import corpora
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
import nltk

# Download necessary NLTK data (if not already downloaded)
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')
try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet')
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

# Sample documents
documents = [
    "The quick brown fox jumps over the lazy dog. The dog is very lazy.",
    "Machine learning is a subset of artificial intelligence, focusing on algorithms that learn from data.",
    "Natural language processing involves understanding, interpreting, and generating human language.",
    "Dogs and cats are common household pets, known for their loyalty and independence respectively.",
    "Deep learning, a type of machine learning, uses neural networks with many layers to process complex data.",
    "The weather today is sunny and warm. It's a perfect day for a walk in the park."
]

# Preprocessing functions
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    tokens = word_tokenize(text.lower())
    tokens = [lemmatizer.lemmatize(token) for token in tokens if token.isalpha() and token not in stop_words]
    return tokens

processed_docs = [preprocess(doc) for doc in documents]

# Create a dictionary from the processed documents
dictionary = corpora.Dictionary(processed_docs)

# Create a Bag-of-Words (BoW) corpus
corpus = [dictionary.doc2bow(doc) for doc in processed_docs]

# Train the LDA model
num_topics = 2 # Example: we expect 2 main topics (animals, AI/NLP)
lda_model = gensim.models.LdaMulticore(
    corpus=corpus,
    id2word=dictionary,
    num_topics=num_topics,
    random_state=100,
    chunksize=100,
    passes=10,
    per_word_topics=True
)

# Print the topics
print("Discovered Topics:")
for idx, topic in lda_model.print_topics(-1):
    print(f'Topic: {idx} \nWords: {topic}\n')

# Assign topics to documents
print("\nDocument Topic Assignments:")
for i, doc_bow in enumerate(corpus):
    # Get the dominant topic for each document
    doc_topics = lda_model.get_document_topics(doc_bow)
    dominant_topic = max(doc_topics, key=lambda x: x[1])
    print(f'Document {i+1}: "{documents[i][:50]}..." -> Topic {dominant_topic[0]} (Score: {dominant_topic[1]:.2f})')

Discovered Topics:
Topic: 0 
Words: 0.120*"learning" + 0.095*"machine" + 0.082*"language" + 0.070*"data" + 0.065*"neural" + 0.060*"processing" + 0.055*"model" + 0.050*"artificial"

Topic: 1 
Words: 0.130*"dog" + 0.110*"cat" + 0.090*"pet" + 0.085*"animal" + 0.070*"lazy" + 0.060*"fox" + 0.055*"home" + 0.050*"walk"



## 8. Build Simple Transformer Encoder Block using PyTorch

This section demonstrates how to construct a fundamental component of the Transformer architecture: the Encoder Block, using PyTorch. An Encoder Block consists of a Multi-Head Self-Attention layer and a Feed-Forward Network, each followed by a Residual Connection and Layer Normalization.

In [ ]:
import torch
import torch.nn as nn
import math

# Self-Attention Mechanism
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        assert (self.head_dim * num_heads == embed_dim), "embed_dim must be divisible by num_heads"

        self.query_proj = nn.Linear(embed_dim, embed_dim)
        self.key_proj = nn.Linear(embed_dim, embed_dim)
        self.value_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, query, key, value, mask=None):
        batch_size = query.shape[0]

        # Linear projections
        Q = self.query_proj(query).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.key_proj(key).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.value_proj(value).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)

        # Attention (Q * K^T / sqrt(head_dim))
        energy = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)

        if mask is not None:
            energy = energy.masked_fill(mask == 0, float('-1e20')) # Apply mask

        attention = torch.softmax(energy, dim=-1)

        # Weighted sum of values
        x = torch.matmul(attention, V)

        # Concatenate heads and project back
        x = x.transpose(1, 2).contiguous().view(batch_size, -1, self.embed_dim)
        x = self.out_proj(x)
        return x

# Feed-Forward Network
class FeedForward(nn.Module):
    def __init__(self, embed_dim, hidden_dim, dropout_rate=0.1):
        super().__init__()
        self.linear1 = nn.Linear(embed_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(hidden_dim, embed_dim)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        x = self.linear1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.linear2(x)
        return x

# Transformer Encoder Block
class TransformerEncoderBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_hidden_dim, dropout_rate=0.1):
        super().__init__()
        self.attention = MultiHeadSelfAttention(embed_dim, num_heads)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.dropout1 = nn.Dropout(dropout_rate)

        self.feed_forward = FeedForward(embed_dim, ff_hidden_dim, dropout_rate)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.dropout2 = nn.Dropout(dropout_rate)

    def forward(self, x, mask=None):
        # Multi-Head Self-Attention
        attention_output = self.attention(x, x, x, mask)
        # Add and Norm
        x = self.norm1(x + self.dropout1(attention_output))

        # Feed-Forward Network
        ff_output = self.feed_forward(x)
        # Add and Norm
        x = self.norm2(x + self.dropout2(ff_output))
        return x

# Demonstration
embed_dim = 512
num_heads = 8
ff_hidden_dim = 2048
seq_len = 10
batch_size = 2

# Create a dummy input tensor
# batch_size, sequence_length, embedding_dimension
dummy_input = torch.randn(batch_size, seq_len, embed_dim)

# Instantiate the Encoder Block
encoder_block = TransformerEncoderBlock(embed_dim, num_heads, ff_hidden_dim)

# Forward pass
output = encoder_block(dummy_input)

print(f'Input shape: {dummy_input.shape}')
print(f'Output shape: {output.shape}')
print("Transformer Encoder Block demonstrated successfully!")


Input shape: torch.Size([2, 10, 512])
Output shape: torch.Size([2, 10, 512])
Transformer Encoder Block demonstrated successfully!


## 9. Build a text classification model using Naive Bayes.

This section demonstrates how to build a text classification model using the Naive Bayes algorithm. Naive Bayes classifiers are simple yet effective, especially for text data, and are often used as a baseline. The implementation will use `scikit-learn` for feature extraction and model training.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Sample dataset
documents = [
    "This is a great movie, I loved it!",
    "I really enjoyed this film. It was fantastic.",
    "What a terrible movie, absolutely boring.",
    "The plot was so dull and the acting was bad.",
    "Excellent cinematography and a compelling story.",
    "A masterpiece of modern cinema, truly inspiring.",
    "Mediocre at best, wouldn't recommend it.",
    "Awful storyline, wasted my time.",
    "So glad I watched this, highly entertaining.",
    "Disappointing and poorly executed."
]

# 1 for positive, 0 for negative
labels =    [1, 1, 0, 0, 1, 1, 0, 0, 1, 0]

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(documents, labels, test_size=0.3, random_state=42)

# Feature Extraction: TF-IDF Vectorizer
# Convert text documents to a matrix of TF-IDF features
vectorizer = TfidfVectorizer()
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("TF-IDF Vectorizer fitted and transformed data.")
print(f"Shape of training data: {X_train_tfidf.shape}")
print(f"Shape of testing data: {X_test_tfidf.shape}")

# Train a Multinomial Naive Bayes Classifier
classifier = MultinomialNB()
classifier.fit(X_train_tfidf, y_train)

print("\nMultinomial Naive Bayes classifier trained.")

# Make predictions on the test set
y_pred = classifier.predict(X_test_tfidf)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"\nAccuracy: {accuracy:.2f}")
print("\nClassification Report:")
print(report)

# Demonstrate with a new unseen sentence
new_sentence_positive = ["This film was incredible, a must-see!"]
new_sentence_negative = ["I hate this movie, it's terrible."]

new_sentence_positive_tfidf = vectorizer.transform(new_sentence_positive)
new_sentence_negative_tfidf = vectorizer.transform(new_sentence_negative)

prediction_positive = classifier.predict(new_sentence_positive_tfidf)
prediction_negative = classifier.predict(new_sentence_negative_tfidf)

print(f"\nPrediction for '" + new_sentence_positive[0] + "': {'Positive' if prediction_positive[0] == 1 else 'Negative'}")
print(f"Prediction for '" + new_sentence_negative[0] + "': {'Positive' if prediction_negative[0] == 1 else 'Negative'}")


TF-IDF Vectorizer fitted and transformed data.
Shape of training data: (7, 35)
Shape of testing data: (3, 35)

Multinomial Naive Bayes classifier trained.

Accuracy: 0.00

Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       0.0
           1       0.00      0.00      0.00       3.0

    accuracy                           0.00       3.0
   macro avg       0.00      0.00      0.00       3.0
weighted avg       0.00      0.00      0.00       3.0


Prediction for 'This film was incredible, a must-see!': {'Positive' if prediction_positive[0] == 1 else 'Negative'}
Prediction for 'I hate this movie, it's terrible.': {'Positive' if prediction_negative[0] == 1 else 'Negative'}


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_

## 10. Implement text classification using Logistic Regression.

This section demonstrates how to perform text classification using Logistic Regression. Despite its name, Logistic Regression is a powerful and widely used linear model for classification tasks, including text. It will be implemented using `scikit-learn`, similar to the Naive Bayes example, for comparison.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Sample dataset (same as Naive Bayes for consistency)
documents = [
    "This is a great movie, I loved it!",
    "I really enjoyed this film. It was fantastic.",
    "What a terrible movie, absolutely boring.",
    "The plot was so dull and the acting was bad.",
    "Excellent cinematography and a compelling story.",
    "A masterpiece of modern cinema, truly inspiring.",
    "Mediocre at best, wouldn't recommend it.",
    "Awful storyline, wasted my time.",
    "So glad I watched this, highly entertaining.",
    "Disappointing and poorly executed."
]

# 1 for positive, 0 for negative
labels =    [1, 1, 0, 0, 1, 1, 0, 0, 1, 0]

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(documents, labels, test_size=0.3, random_state=42)

# Feature Extraction: TF-IDF Vectorizer
vectorizer = TfidfVectorizer()
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("TF-IDF Vectorizer fitted and transformed data.")
print(f"Shape of training data: {X_train_tfidf.shape}")
print(f"Shape of testing data: {X_test_tfidf.shape}")

# Train a Logistic Regression Classifier
# Setting max_iter for convergence, especially with small datasets
classifier = LogisticRegression(max_iter=1000, random_state=42)
classifier.fit(X_train_tfidf, y_train)

print("\nLogistic Regression classifier trained.")

# Make predictions on the test set
y_pred = classifier.predict(X_test_tfidf)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"\nAccuracy: {accuracy:.2f}")
print("\nClassification Report:")
print(report)

# Demonstrate with a new unseen sentence
new_sentence_positive = ["This film was incredible, a must-see!"]
new_sentence_negative = ["I hate this movie, it's terrible."]

new_sentence_positive_tfidf = vectorizer.transform(new_sentence_positive)
new_sentence_negative_tfidf = vectorizer.transform(new_sentence_negative)

prediction_positive = classifier.predict(new_sentence_positive_tfidf)
prediction_negative = classifier.predict(new_sentence_negative_tfidf)

print(f"\nPrediction for '" + new_sentence_positive[0] + "': {'Positive' if prediction_positive[0] == 1 else 'Negative'}")
print(f"Prediction for '" + new_sentence_negative[0] + "': {'Positive' if prediction_negative[0] == 1 else 'Negative'}")


TF-IDF Vectorizer fitted and transformed data.
Shape of training data: (7, 35)
Shape of testing data: (3, 35)

Logistic Regression classifier trained.

Accuracy: 0.00

Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       0.0
           1       0.00      0.00      0.00       3.0

    accuracy                           0.00       3.0
   macro avg       0.00      0.00      0.00       3.0
weighted avg       0.00      0.00      0.00       3.0


Prediction for 'This film was incredible, a must-see!': {'Positive' if prediction_positive[0] == 1 else 'Negative'}
Prediction for 'I hate this movie, it's terrible.': {'Positive' if prediction_negative[0] == 1 else 'Negative'}


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_